
# TP — Classification : Régression Logistique et k-NN

**Niveau :** Master  
**Durée :** 2 heures  
Généré le : 2026-03-12

Ce TP contient **cinq parties** :

1. **Régression Logistique Binaire implémentée à la main**
2. **Régression Logistique Multiclasse (Softmax) implémentée à la main**
3. **Utilisation des modèles scikit-learn et comparaison (Régression Logistique vs k-NN)**
4. **Jeu de données Moons : limites de la régression logistique et avantage du k-NN**
5. **Jeu de données Cancer du Sein : du modèle à 2 variables à l'évaluation clinique complète**

Jeux de données :
- Jeu de données synthétique (pour l'implémentation à la main)
- Jeu de données Titanic (classification binaire)
- Jeu de données Penguins (classification multiclasse)
- Jeu de données Cancer du Sein Wisconsin



# Partie 1 — Régression Logistique Binaire (Implémentée à la Main)

Dans cette section, nous implémentons la **régression logistique from scratch** avec la descente de gradient.

Le modèle logistique estime la probabilité :

$$
P(Y=1|x)=\sigma(w^Tx+b)
$$

où la **fonction sigmoïde** est

$$
\sigma(z)=\frac{1}{1+e^{-z}}
$$

Pour entraîner le modèle, on minimise la **perte d'entropie croisée binaire** :

$$
L = - \frac{1}{N} \sum_{i=1}^N
[y_i \log(p_i) + (1-y_i)\log(1-p_i)]
$$


In [ ]:
import numpy as np
import matplotlib.pyplot as plt


## Générer un jeu de données binaire simple

In [ ]:

np.random.seed(0)

N = 200

X = np.random.randn(N,2)

true_w = np.array([2,-1])

logits = X @ true_w

y = (logits > 0).astype(int)


## Fonction sigmoïde

In [ ]:

def sigmoid(z):
    return 1/(1+np.exp(-z))


## Perte d'entropie croisée binaire et gradient

In [ ]:

def binary_cross_entropy(X,y,w):

    z = X @ w
    p = sigmoid(z)

    loss = -np.mean(
        y*np.log(p+1e-9) + (1-y)*np.log(1-p+1e-9)
    )

    grad = X.T @ (p-y) / len(y)

    return loss, grad


## Optimiseur par descente de gradient

In [ ]:

def gradient_descent(X,y,lr=0.1,epochs=200):

    w = np.zeros(X.shape[1])

    losses=[]

    for _ in range(epochs):

        loss, grad = binary_cross_entropy(X,y,w)

        w = w - lr*grad

        losses.append(loss)

    return w, losses


In [ ]:

w, losses = gradient_descent(X,y)

plt.plot(losses)
plt.title("Perte d'entraînement")
plt.show()



---

# Partie 2 — Régression Logistique Multiclasse (Softmax) Implémentée à la Main

Pour **K classes**, la régression logistique utilise la **fonction softmax**.

Softmax converts logits into probabilities:

$$
softmax(z_i) =
\frac{e^{z_i}}{\sum_{j=1}^{K} e^{z_j}}
$$

Each class receives a probability and the probabilities sum to 1.


In [ ]:

def softmax(z):
    exp = np.exp(z - np.max(z,axis=1,keepdims=True))
    return exp/exp.sum(axis=1,keepdims=True)



## Perte d'Entropie Croisée Multiclasse

Pour la classification multiclasse, la perte devient :

$$
L = - \frac{1}{N} \sum_{i=1}^{N}
\sum_{k=1}^{K} y_{ik} \log(p_{ik})
$$


In [ ]:

def multiclass_cross_entropy(X,Y,W):

    logits = X @ W

    P = softmax(logits)

    loss = -np.mean(np.sum(Y*np.log(P+1e-9),axis=1))

    grad = X.T @ (P-Y) / len(X)

    return loss, grad


## Générer un jeu de données multiclasse jouet

In [ ]:

np.random.seed(1)

N = 300
D = 2
K = 3

X = np.random.randn(N,D)

W_true = np.random.randn(D,K)

logits = X @ W_true

y = np.argmax(logits,axis=1)

Y = np.eye(K)[y]


## Entraîner avec la descente de gradient

In [ ]:

def train_softmax(X,Y,lr=0.1,epochs=300):

    W = np.zeros((X.shape[1],Y.shape[1]))

    losses=[]

    for _ in range(epochs):

        loss,grad = multiclass_cross_entropy(X,Y,W)

        W -= lr*grad

        losses.append(loss)

    return W, losses


In [ ]:

W, losses = train_softmax(X,Y)

plt.plot(losses)
plt.title("Perte d'entraînement multiclasse")
plt.show()



---

# Partie 3 — Utilisation des Modèles Scikit-Learn

Nous appliquons maintenant la **Régression Logistique** et les **k-Plus Proches Voisins** à des jeux de données réels.


In [ ]:

import pandas as pd
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import confusion_matrix, precision_score, recall_score


## Jeu de données Titanic (Classification Binaire)

Nous commençons par une tâche binaire : prédire la survie sur le Titanic.

In [ ]:
# Chargement du jeu Titanic et sélection des variables numériques
# Suppression des lignes avec valeurs manquantes
titanic = sns.load_dataset("titanic")

features = ['pclass','age','fare','sibsp','parch']

df = titanic[features+['survived']].dropna()

X = df[features]
y = df['survived']

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=0)

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


In [ ]:
# Train Logistic Regression on Titanic and print confusion matrix
log_model = LogisticRegression(max_iter=1000)

log_model.fit(X_train,y_train)

pred = log_model.predict(X_test)

print(confusion_matrix(y_test,pred))


In [ ]:
# Train k-NN on Titanic and print confusion matrix
knn = KNeighborsClassifier(n_neighbors=5)

knn.fit(X_train,y_train)

pred_knn = knn.predict(X_test)

print(confusion_matrix(y_test,pred_knn))


## Jeu de données Penguins (Classification Multiclasse)

Maintenant une tâche multiclasse : prédire l'espèce de pingouin à partir de mesures morphologiques.

In [ ]:
# Chargement du jeu Penguins — 3 espèces, 4 variables numériques
# Suppression des lignes avec valeurs manquantes
penguins = sns.load_dataset("penguins").dropna()

X = penguins[['bill_length_mm','bill_depth_mm','flipper_length_mm','body_mass_g']]
y = penguins['species']

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=0)

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


In [ ]:
# Train sklearn Logistic Regression (multinomial) on Penguins
log_model = LogisticRegression(max_iter=1000)

log_model.fit(X_train,y_train)

pred = log_model.predict(X_test)

print("Régression Logistique Sklearn — matrice de confusion :")
print(confusion_matrix(y_test,pred))
print(f"Précision : {log_model.score(X_test, y_test):.3f}")


In [ ]:
# Train k-NN on Penguins and print confusion matrix
knn = KNeighborsClassifier(n_neighbors=5)

knn.fit(X_train,y_train)

pred_knn = knn.predict(X_test)

print("k-NN — matrice de confusion :")
print(confusion_matrix(y_test,pred_knn))
print(f"Précision : {knn.score(X_test, y_test):.3f}")


## Comparaison entre notre Softmax fait à la main et Sklearn

Nous comparons maintenant la régression logistique multiclasse construite en Partie 2
avec l'implémentation sklearn — sur les mêmes données Penguins.


In [ ]:
# Ré-encoder les espèces en entiers pour notre implémentation
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

import numpy as np
X_raw = penguins[['bill_length_mm','bill_depth_mm','flipper_length_mm','body_mass_g']].values
y_raw = le.fit_transform(penguins['species'].values)

sc = StandardScaler()
X_scaled = sc.fit_transform(X_raw)

X_tr, X_te, y_tr, y_te = train_test_split(X_scaled, y_raw, test_size=0.2, random_state=0)

# One-hot encode for our softmax
K = len(np.unique(y_raw))
Y_tr = np.eye(K)[y_tr]

# Train our hand-implemented softmax
W_penguins, losses_penguins = train_softmax(X_tr, Y_tr, lr=0.1, epochs=500)

# Predict with our model
logits_te = X_te @ W_penguins
y_pred_hand = np.argmax(logits_te, axis=1)
acc_hand = np.mean(y_pred_hand == y_te)

# Predict with sklearn
log_sk = LogisticRegression(max_iter=1000)
log_sk.fit(X_tr, y_tr)
acc_sklearn = log_sk.score(X_te, y_te)

print(f"Précision Softmax à la main       : {acc_hand:.3f}")
print(f"Précision Régression Logistique Sklearn : {acc_sklearn:.3f}")


In [ ]:
# Plot training loss curve of our hand implementation on Penguins
plt.figure(figsize=(6,3))
plt.plot(losses_penguins, color='steelblue')
plt.title("Softmax fait à la main — perte d'entraînement sur Penguins")
plt.xlabel("Époque")
plt.ylabel("Perte")
plt.tight_layout()
plt.show()

# Confusion matrix comparison side by side
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

species_names = le.classes_.tolist()

for ax, preds, title in zip(
    axes,
    [y_pred_hand, log_sk.predict(X_te)],
    ["Softmax à la main", "Régression Logistique Sklearn"]
):
    cm = confusion_matrix(y_te, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=species_names, yticklabels=species_names)
    ax.set_xlabel("Prédit")
    ax.set_ylabel("Réel")
    ax.set_title(title)

plt.tight_layout()
plt.show()



---

## Quand la Régression Logistique Échoue — Données Non Linéairement Séparables (Jeu Moons)

La régression logistique suppose une **frontière de décision linéaire**.
Quand les classes ne sont pas linéairement séparables, elle ne peut pas bien s'adapter aux données.
k-NN ne fait **pas cette hypothèse** et peut capturer des frontières courbes et complexes.

Nous utilisons le jeu `make_moons` — deux demi-cercles imbriqués — comme illustration.


In [ ]:
# Générer le jeu moons
# noise contrôle le chevauchement entre les deux classes
from sklearn.datasets import make_moons

X_moons, y_moons = make_moons(n_samples=300, noise=0.2, random_state=0)

# Visualiser les données brutes
plt.figure(figsize=(6, 4))
plt.scatter(X_moons[:, 0], X_moons[:, 1], c=y_moons, cmap='bwr', edgecolors='k', s=30)
plt.title("Jeu Moons — non linéairement séparable")
plt.tight_layout()
plt.show()

In [ ]:
# Division train/test — pas besoin de normalisation ici (variables déjà à la même échelle)
X_train_m, X_test_m, y_train_m, y_test_m = train_test_split(
    X_moons, y_moons, test_size=0.2, random_state=0
)

log_moons = LogisticRegression()
knn_moons = KNeighborsClassifier(n_neighbors=5)

log_moons.fit(X_train_m, y_train_m)
knn_moons.fit(X_train_m, y_train_m)

In [ ]:
# Tracer les frontières de décision côte à côte — la visualisation clé de cette section
from sklearn.inspection import DecisionBoundaryDisplay

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, model, name in zip(
    axes,
    [log_moons, knn_moons],
    ["Régression Logistique", "k-NN (k=5)"]
):
    DecisionBoundaryDisplay.from_estimator(
        model, X_moons, cmap="bwr", alpha=0.3, ax=ax
    )
    ax.scatter(X_moons[:, 0], X_moons[:, 1], c=y_moons,
               cmap="bwr", edgecolors='k', s=30)
    acc = model.score(X_test_m, y_test_m)
    ax.set_title(f"{name}\nPrécision test : {acc:.2f}")

plt.tight_layout()
plt.show()

In [ ]:
# Afficher les précisions côte à côte
print(f"Précision Régression Logistique : {log_moons.score(X_test_m, y_test_m):.2f}")
print(f"Précision k-NN               : {knn_moons.score(X_test_m, y_test_m):.2f}")


---

# Partie 4 — Cancer du Sein : Du Modèle à 2 Variables au Modèle Complet

Cette partie suit une **progression pédagogique** :

1. Identifier les **2 variables les plus discriminantes** en explorant les corrélations
2. Implémenter la régression logistique **à la main** sur ces 2 variables → visualiser la frontière de décision
3. Entraîner Régression Logistique et k-NN sklearn sur **les 30 variables** → comparer les performances
4. Discuter **précision vs rappel** — pourquoi l'exactitude seule est trompeuse en contexte médical


## Étape 1 — Charger le jeu de données et explorer l'équilibre des classes

In [ ]:
# Charger le jeu de données Cancer du Sein Wisconsin
# 569 échantillons, 30 variables
# Cible : 0 = malin, 1 = bénin
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer()
df_bc = pd.DataFrame(data.data, columns=data.feature_names)
df_bc['target'] = data.target

print("Dimensions :", df_bc.shape)
print("\nDistribution des classes :")
print(df_bc['target'].value_counts().rename({0: "malin", 1: "bénin"}))


## Étape 2 — Trouver les 2 variables les plus discriminantes

In [ ]:
# Calculer la corrélation absolue de chaque variable avec la cible
# Corrélation absolue élevée = plus discriminante
correlations = df_bc.drop(columns='target').corrwith(df_bc['target']).abs().sort_values(ascending=False)

print("Top 10 variables par corrélation avec la cible :")
print(correlations.head(10).round(3))


In [ ]:
# Tracer les 15 premières corrélations en diagramme à barres
# Aide à identifier visuellement les variables les plus utiles
plt.figure(figsize=(10, 4))
correlations.head(15).plot(kind='bar', color='steelblue', edgecolor='k')
plt.title("Corrélation des variables avec la cible (malin vs bénin)")
plt.ylabel("Corrélation absolue")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


In [ ]:
# Extraire les 2 meilleures variables
# Elles seront utilisées pour la régression logistique à la main
top2 = correlations.head(2).index.tolist()
print("Variables sélectionnées :", top2)

X2 = df_bc[top2].values
y2 = df_bc['target'].values


## Étape 3 — Visualiser l'espace à 2 variables

In [ ]:
# Nuage de points des deux variables sélectionnées
# Couleur = classe (malin=rouge, bénin=bleu)
# Permet de voir si une frontière linéaire semble raisonnable
plt.figure(figsize=(7, 5))
for label, color, name in zip([0, 1], ['red', 'steelblue'], ['malignant', 'benign']):
    mask = y2 == label
    plt.scatter(X2[mask, 0], X2[mask, 1],
                c=color, label=name, alpha=0.5, edgecolors='k', s=30)

plt.xlabel(top2[0])
plt.ylabel(top2[1])
plt.title("Cancer du Sein — 2 variables les plus discriminantes")
plt.legend()
plt.tight_layout()
plt.show()


## Étape 4 — Régression Logistique à la Main sur 2 Variables

Nous réutilisons les fonctions `sigmoid`, `binary_cross_entropy` et `gradient_descent` de la Partie 1.


In [ ]:
# Réutiliser les fonctions de la Partie 1 — s'assurer que les cellules de la Partie 1 ont été exécutées
# Normaliser les 2 variables avant la descente de gradient
scaler2 = StandardScaler()
X2_scaled = scaler2.fit_transform(X2)

X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2_scaled, y2, test_size=0.2, random_state=0
)


In [ ]:
# Entraîner la régression logistique à la main sur les 2 variables sélectionnées
w2, losses2 = gradient_descent(X2_train, y2_train, lr=0.1, epochs=500)

plt.figure(figsize=(6, 3))
plt.plot(losses2, color='steelblue')
plt.title("Perte d'entraînement — régression logistique à la main (2 variables)")
plt.xlabel("Époque")
plt.ylabel("Perte")
plt.tight_layout()
plt.show()


In [ ]:
# Évaluer la précision sur l'ensemble de test
p2_test = sigmoid(X2_test @ w2)
y2_pred = (p2_test >= 0.5).astype(int)

acc2 = np.mean(y2_pred == y2_test)
print(f"Précision régression logistique à la main (2 variables) : {acc2:.3f}")


## Étape 5 — Tracer la frontière de décision (2 variables)

In [ ]:
# Tracer la frontière de décision linéaire apprise à la main
# La frontière vérifie w0*x0 + w1*x1 = 0  →  x1 = -(w0/w1)*x0
x0_range = np.linspace(X2_scaled[:, 0].min() - 0.5,
                        X2_scaled[:, 0].max() + 0.5, 200)
x1_boundary = -(w2[0] / w2[1]) * x0_range

plt.figure(figsize=(7, 5))
for label, color, name in zip([0, 1], ['red', 'steelblue'], ['malignant', 'benign']):
    mask = y2 == label
    plt.scatter(X2_scaled[mask, 0], X2_scaled[mask, 1],
                c=color, label=name, alpha=0.5, edgecolors='k', s=30)

plt.plot(x0_range, x1_boundary, 'k--', lw=2, label='Frontière de décision')
plt.xlabel(top2[0] + " (scaled)")
plt.ylabel(top2[1] + " (scaled)")
plt.title("Régression logistique à la main — frontière de décision")
plt.legend()
plt.tight_layout()
plt.show()


## Étape 6 — Modèles Complets avec les 30 Variables (sklearn)

Nous utilisons maintenant les 30 variables et comparons Régression Logistique et k-NN.


In [ ]:
# Préparer le jeu de données complet avec les 30 variables
X_full = data.data
y_full = data.target

X_train_f, X_test_f, y_train_f, y_test_f = train_test_split(
    X_full, y_full, test_size=0.2, random_state=0
)

scaler_f = StandardScaler()
X_train_f = scaler_f.fit_transform(X_train_f)
X_test_f  = scaler_f.transform(X_test_f)


In [ ]:
# Entraîner les deux modèles sur les 30 variables et comparer les précisions
log_full = LogisticRegression(max_iter=1000)
knn_full = KNeighborsClassifier(n_neighbors=5)

log_full.fit(X_train_f, y_train_f)
knn_full.fit(X_train_f, y_train_f)

for name, model in [("Régression Logistique (30 variables)", log_full),
                    ("k-NN              (30 variables)", knn_full)]:
    acc = model.score(X_test_f, y_test_f)
    print(f"{name} | Précision : {acc:.3f}")


In [ ]:
# Matrices de confusion côte à côte
# Lignes = classe réelle, colonnes = classe prédite
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, model, name in zip(
    axes,
    [log_full, knn_full],
    ["Régression Logistique", "k-NN (k=5)"]
):
    cm = confusion_matrix(y_test_f, model.predict(X_test_f))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['malin', 'bénin'],
                yticklabels=['malin', 'bénin'])
    ax.set_xlabel("Prédit")
    ax.set_ylabel("Réel")
    ax.set_title(name)

plt.tight_layout()
plt.show()


## Étape 7 — Précision vs Rappel : Pourquoi l'Exactitude ne Suffit Pas

Dans un contexte médical, les deux types d'erreurs ne sont **pas équivalents** :

| Erreur | Signification | Conséquence |
|---|---|---|
| **Faux Négatif** | Prédit bénin, en réalité malin | ⚠️ Patient non traité |
| **Faux Positif** | Prédit malin, en réalité bénin | Examens supplémentaires, stress |

Un **faux négatif est bien plus dangereux** ici.  
Cela signifie qu'on doit privilégier le **rappel** (sensibilité) par rapport à la précision pour la classe maligne.

$$\text{Recall} = \frac{TP}{TP + FN}$$

$$\text{Precision} = \frac{TP}{TP + FP}$$


In [ ]:
# Afficher le rapport de classification complet pour les deux modèles
# Attention au rappel pour la classe 0 (malin)
from sklearn.metrics import classification_report

for name, model in [("Régression Logistique", log_full), ("k-NN", knn_full)]:
    print(f"=== {name} ===")
    print(classification_report(
        y_test_f,
        model.predict(X_test_f),
        target_names=['malin (0)', 'bénin (1)']
    ))


## Étape 8 — Courbe ROC et Score AUC

La **courbe ROC** montre comment le compromis précision/rappel évolue quand on fait varier le seuil de décision par rapport à la valeur par défaut 0.5.

**AUC (Area Under the Curve):** a single number summarizing model quality across all thresholds.  
$AUC = 1$ est parfait, $AUC = 0.5$ est aléatoire.


In [ ]:
# Tracer les courbes ROC pour les deux modèles
# Plus la courbe se rapproche du coin supérieur gauche, meilleur est le modèle
from sklearn.metrics import roc_curve, roc_auc_score

plt.figure(figsize=(7, 5))

for model, name, color in [
    (log_full, "Régression Logistique", "steelblue"),
    (knn_full, "k-NN (k=5)",          "tomato")
]:
    proba = model.predict_proba(X_test_f)[:, 1]
    fpr, tpr, _ = roc_curve(y_test_f, proba)
    auc = roc_auc_score(y_test_f, proba)
    plt.plot(fpr, tpr, lw=2, color=color, label=f"{name}  (AUC = {auc:.3f})")

plt.plot([0, 1], [0, 1], 'k--', label='Classifieur aléatoire')
plt.xlabel("Taux de Faux Positifs")
plt.ylabel("Taux de Vrais Positifs (Rappel)")
plt.title("Courbe ROC — Cancer du Sein")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Que se passe-t-il si on baisse le seuil de décision de 0.5 à 0.3 ?
# → On détecte plus de cas malins (rappel plus élevé) mais on classe plus de bénins comme malins
threshold = 0.3

proba_log = log_full.predict_proba(X_test_f)[:, 1]
y_pred_adjusted = (proba_log >= threshold).astype(int)

print(f"=== Régression Logistique avec seuil = {threshold} ===")
print(classification_report(
    y_test_f,
    y_pred_adjusted,
    target_names=['malin (0)', 'bénin (1)']
))


## Résumé — Ce que nous avons appris en Partie 4

| Étape | Point clé |
|---|---|
| Modèle à 2 variables | Une frontière linéaire est visible et interprétable |
| Modèle complet 30 variables | Les performances augmentent significativement |
| Logistique vs k-NN | Les deux sont bons ; la régression logistique est plus interprétable |
| Précision vs Rappel | En médecine, **le rappel sur la classe dangereuse prime** |
| ROC / AUC | Meilleure évaluation que l'exactitude pour les tâches déséquilibrées ou à enjeux |
| Ajustement du seuil | Baisser le seuil augmente le rappel au détriment de la précision |
